# ?? Module 01: FinQA Exploratory Data Analysis & SQLite Ingestion
**Project:** AI-Driven Financial Planning & Risk Management System  
**Purpose:** Kh?m ph? c?u tr?c d? li?u t?i ch?nh FinQA (b?ng bi?u + v?n b?n + c?u h?i + chu?i t?nh to?n), gi?i quy?t c?c b?i to?n h?ng kh?ng ??ng ??u (*uneven rows*), v? n?p t? ??ng 100 b?ng ??u ti?n v?o c? s? d? li?u quan h? **SQLite (`data/database/finance.db`)** ph?c v? b?i to?n **Text-to-SQL & Risk Guardrails**.

## 1. Kh?i t?o m?i tr??ng & Import th? vi?n

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text, inspect

# Th?m th? m?c g?c v?o sys.path ?? import c?c module src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.data_parser import parse_raw_table_to_df, parse_finqa_to_sqlite
print(f"Project root: {project_root}")
print("Libraries imported successfully!")

## 2. Kh?m ph? c?u tr?c FinQA (`train.json`)
M?i m?u d? li?u trong FinQA bao g?m:
- `id`: M? ??nh danh b?o c?o (m? c? phi?u / n?m / trang pdf)
- `pre_text` & `post_text`: ?o?n v?n b?n thuy?t minh tr??c v? sau b?ng s? li?u
- `table` & `table_ori`: D? li?u b?ng bi?u d?ng m?ng 2 chi?u (g?c v? ?? token h?a)
- `qa`: Ch?a `question`, `program` (chu?i h?m t?nh to?n s? h?c), `exe_ans` (k?t qu? cu?i c?ng)

In [ ]:
train_path = project_root / "FinQA" / "dataset" / "train.json"
if not train_path.exists():
    train_path = project_root / "FinQA" / "train.json"

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print(f"T?ng s? m?u trong t?p train: {len(train_data):,}")
sample = train_data[0]

print("\n--- M?U D? LI?U ??U TI?N ---")
print(f"ID: {sample['id']}")
print(f"File b?o c?o: {sample['filename']}")
print(f"C?u h?i t?i ch?nh: {sample['qa']['question']}")
print(f"Ch??ng tr?nh t?nh to?n (Gold Program): {sample['qa']['program']}")
print(f"K?t qu? th?c thi (Execution Answer): {sample['qa']['exe_ans']}")

### Chi ti?t B?ng Bi?u (Table Visualization)

In [ ]:
# Hi?n th? d? li?u b?ng th?
raw_table = sample.get("table_ori", sample.get("table"))
print(f"S? h?ng trong b?ng: {len(raw_table)}")

# Chuy?n ??i sang pandas DataFrame b?ng module chu?n h?a c?a d? ?n
sample_df = parse_raw_table_to_df(raw_table, sample_id=sample['id'])
display(sample_df)

## 3. N?p 100 b?ng FinQA v?o c? s? d? li?u SQLite (`finance.db`)
H?m `parse_finqa_to_sqlite` th?c hi?n:
1. L?p qua 100 b?ng ??u ti?n trong `train.json`.
2. X? l? c?c h?ng c? ?? d?i kh?ng ??ng ??u (*uneven row padding*).
3. Chu?n h?a t?n c?t h?p l? cho chu?n SQL (lo?i b? k? t? ??c bi?t, ch?ng tr?ng l?p t?n c?t).
4. L?u t?ng b?ng th?nh `table_0`, `table_1`, ..., `table_99`.
5. L?u to?n b? metadata (c?u h?i, c?ng th?c, ??p ?n, filename) v?o b?ng `finqa_metadata`.

In [ ]:
db_path = project_root / "data" / "database" / "finance.db"

summary = parse_finqa_to_sqlite(
    json_path=str(train_path),
    db_path=str(db_path),
    max_tables=100,
    use_table_ori=True,
    if_exists="replace"
)

print("\nK?t qu? n?p CSDL:")
print(json.dumps(summary, indent=2))

## 4. Truy v?n & Ki?m tra CSDL SQLite b?ng SQLAlchemy

In [ ]:
db_uri = f"sqlite:///{db_path.resolve().as_posix()}"
engine = create_engine(db_uri)

# Ki?m tra danh s?ch b?ng ?? t?o
inspector = inspect(engine)
tables = inspector.get_table_names()
print(f"T?ng s? b?ng trong finance.db: {len(tables)}")
print("5 b?ng ??u ti?n:", tables[:5])
print("B?ng metadata:", [t for t in tables if "metadata" in t])

### ??c th?ng tin t? b?ng `finqa_metadata`

In [ ]:
metadata_query = """
SELECT table_id, sample_id, question, program, exe_ans, num_rows, num_cols 
FROM finqa_metadata 
LIMIT 5;
"""

meta_df = pd.read_sql(metadata_query, con=engine)
display(meta_df)

### Truy v?n tr?c ti?p m?t b?ng t?i ch?nh c? th? (`table_0`)

In [ ]:
# Truy v?n to?n b? d? li?u c?a table_0
table_0_df = pd.read_sql("SELECT * FROM table_0;", con=engine)
print("C?u tr?c v? s? li?u b?ng table_0:")
display(table_0_df)

## 5. Th?ng k? ph?n b? s? h?ng v? s? c?t c?a 100 b?ng m?u

In [ ]:
stats_df = pd.read_sql("SELECT num_rows, num_cols FROM finqa_metadata;", con=engine)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Ph?n b? s? h?ng
axes[0].hist(stats_df["num_rows"], bins=15, color="skyblue", edgecolor="black")
axes[0].set_title("Ph?n b? s? h?ng (Number of Rows)")
axes[0].set_xlabel("S? h?ng")
axes[0].set_ylabel("T?n su?t")

# Ph?n b? s? c?t
axes[1].hist(stats_df["num_cols"], bins=10, color="salmon", edgecolor="black")
axes[1].set_title("Ph?n b? s? c?t (Number of Columns)")
axes[1].set_xlabel("S? c?t")
axes[1].set_ylabel("T?n su?t")

plt.tight_layout()
plt.show()

print(f"S? h?ng trung b?nh: {stats_df['num_rows'].mean():.1f}")
print(f"S? c?t trung b?nh: {stats_df['num_cols'].mean():.1f}")

## 6. T?ng k?t & B??c ti?p theo
- ? **?? ho?n th?nh:**
  - Kh?m ph? v? hi?u r? c?u tr?c d? li?u h?n h?p (v?n b?n + b?ng bi?u) c?a FinQA.
  - Chu?n h?a th?nh c?ng c?c b?ng t?i ch?nh v? gi?i quy?t v?n ?? h?ng kh?ng ??u (*uneven row padding*).
  - T?ch h?p t? ??ng 100 b?ng v?o SQLite (`finance.db`) k?m b?ng `finqa_metadata` ??y ?? th?ng tin c?u h?i v? chu?i h?m t?nh to?n.
- ?? **B??c ti?p theo:**
  - Ph?t tri?n module **Text-to-SQL** (`src/text_to_sql/sql_generator.py`) ?? sinh c?u truy v?n SQL t? c?u h?i trong `finqa_metadata`.
  - T?ch h?p **Risk Guardrails** (`src/text_to_sql/risk_guardrails.py`) ng?n ch?n c?c truy v?n r?i ro tr??c khi th?c thi v?o CSDL.